# Vietnamese Handwriting OCR — Demo Streamlit

Notebook bao gồm:
1. Cài đặt môi trường.
2. Tải mã nguồn PaddleOCR và repository giao diện từ Github.
3. Khai báo các đường dẫn và thiết lập cấu hình chạy inference.
4. Chạy giao diện Streamlit tại ngrok tunnel.

> Lưu ý: Đảm bảo đính kèm dataset có chứa model SVTR Stage 2 và CRNN Stage 2 trước khi chạy.

## 1. Cài đặt môi trường PaddlePaddle

In [1]:
import sys, subprocess, os, re

os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'

# ── Uninstall torch để tránh conflict ──────────────────────
subprocess.run([
    sys.executable, '-m', 'pip', 'uninstall', '-y',
    'torch', 'torchvision', 'torchaudio'
], check=False, capture_output=True)

# ── Auto-detect CUDA version ──────────────────────────────
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True, shell=True)
cuda_match = re.search(r'CUDA Version:\s*(\d+)', result.stdout)
cuda_major = int(cuda_match.group(1)) if cuda_match else 11

if cuda_major >= 12:
    paddle_pkg = 'paddlepaddle-gpu==3.0.0'
    paddle_idx = 'https://www.paddlepaddle.org.cn/packages/stable/cu126/'
    print(f'🚀 CUDA {cuda_major}.x → paddle cu126')
else:
    paddle_pkg = 'paddlepaddle-gpu==2.6.2.post118'
    paddle_idx = 'https://www.paddlepaddle.org.cn/packages/stable/cu118/'
    print(f'🐢 CUDA {cuda_major}.x → paddle cu118')

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       paddle_pkg, '-i', paddle_idx])

# ── Cài thư viện bổ sung ───────────────────────────────────
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'streamlit', 'pyngrok', 'lmdb', 'Pillow', 'opencv-python-headless'])

import paddle
print(f'✅ Paddle {paddle.__version__} | GPU: {paddle.device.cuda.device_count()}')

🚀 CUDA 13.x → paddle cu126
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 GB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 9.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 50.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 16.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.0/571.0 MB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 10.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.2/158.2 MB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.6/216.6 MB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.8/156.8 MB 5.7 MB/s eta 0:00:00
     ━━━

/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:711: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


✅ Paddle 3.0.0 | GPU: 2


## 2. Chuẩn bị mã nguồn PaddleOCR và Repository Giao Diện

In [2]:
import os, subprocess
from pathlib import Path

PADDLEOCR_DIR = Path('/kaggle/working/PaddleOCR')
REPO_DIR = Path('/kaggle/working/Recognizing-Vietnamese-handwriting')

if not PADDLEOCR_DIR.is_dir():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/PaddlePaddle/PaddleOCR.git', str(PADDLEOCR_DIR)], check=True)
    print('Cloned PaddleOCR.')
else:
    print('PaddleOCR exists.')

if not REPO_DIR.is_dir():
    subprocess.run(['git', 'clone', 'https://github.com/KhoaLeDang2375/Recognizing-Vietnamese-handwriting.git', str(REPO_DIR)], check=True)
    print('Cloned user repository.')
else:
    print('User repository exists.')

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PADDLEOCR_DIR / 'requirements.txt')])
# ── Patch HandwritingAug (cần thiết nếu config train dùng nó) ──
AUG_FILE = PADDLEOCR_DIR / 'ppocr/data/imaug/handwriting_aug.py'
AUG_CODE = '''
import cv2, numpy as np, random

class HandwritingAug:
    """Lightweight handwriting augmentation for Vietnamese OCR."""
    def __init__(self, aug_prob=0.5, **kw):
        self.aug_prob = float(aug_prob)

    def __call__(self, data):
        if random.random() > self.aug_prob:
            return data
        img = data.get("image")
        if img is None or img.size == 0:
            return data
        try:
            img = self._brightness(img)
            img = self._contrast(img)
            img = self._blur(img)
            img = self._shear(img)
            img = self._random_erase(img)
        except Exception:
            pass
        data["image"] = img
        return data

    def _brightness(self, img):
        f = random.uniform(0.7, 1.3)
        return np.clip(img.astype(np.float32) * f, 0, 255).astype(np.uint8)

    def _contrast(self, img):
        f = random.uniform(0.7, 1.3)
        mean = img.mean()
        return np.clip((img.astype(np.float32) - mean) * f + mean, 0, 255).astype(np.uint8)

    def _blur(self, img):
        if random.random() < 0.3:
            k = random.choice([3, 5])
            img = cv2.GaussianBlur(img, (k, k), 0)
        return img

    def _shear(self, img):
        if random.random() < 0.3:
            h, w = img.shape[:2]
            sh = random.uniform(-0.15, 0.15)
            M = np.float32([[1, sh, 0], [0, 1, 0]])
            img = cv2.warpAffine(img, M, (w, h),
                                 borderMode=cv2.BORDER_REPLICATE)
        return img

    def _random_erase(self, img):
        if random.random() < 0.2:
            h, w = img.shape[:2]
            eh = random.randint(2, max(3, h//5))
            ew = random.randint(5, max(6, w//5))
            y0 = random.randint(0, h - eh)
            x0 = random.randint(0, w - ew)
            img[y0:y0+eh, x0:x0+ew] = 255
        return img
'''

if not AUG_FILE.exists():
    AUG_FILE.write_text(AUG_CODE, encoding='utf-8')
    # Patch __init__.py
    init_file = PADDLEOCR_DIR / 'ppocr/data/imaug/__init__.py'
    init_content = init_file.read_text(encoding='utf-8')
    if 'HandwritingAug' not in init_content:
        init_file.write_text(
            init_content + '\nfrom .handwriting_aug import HandwritingAug\n',
            encoding='utf-8'
        )
    print('✅ HandwritingAug patched')
else:
    print('✅ HandwritingAug đã tồn tại')

print('\n✅ PaddleOCR source sẵn sàng!')

Cloning into '/kaggle/working/PaddleOCR'...


✅ Cloned PaddleOCR
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 33.1 MB/s eta 0:00:00
✅ HandwritingAug patched

✅ PaddleOCR source sẵn sàng!


## 3. Khai báo các đường dẫn hệ thống

In [3]:
from pathlib import Path

# Dataset
DATASET_SLUG = 'datasets/khangphantrnvn/uit-handwriten-dataset'
DATA_ROOT = Path(f'/kaggle/input/{DATASET_SLUG}/UIT_HWDB_line_converted')
TRAIN_DATA = DATA_ROOT / 'train_data'
TEST_DATA = DATA_ROOT / 'test_data'

# Model
MODEL_SLUG = 'models/thoandanh'
MODEL_ROOT = Path(f'/kaggle/input/{MODEL_SLUG}')

# Checkpoint directories (fixed path issue)
SVTR_CKPT_DIR = MODEL_ROOT / 'svtr-vietnamese-handwriten/pytorch/default/1/SVTR/Stage2/best_accuracy'
CRNN_CKPT_DIR = MODEL_ROOT / 'crnn-vietnamese-handwriten/pytorch/default/1/CRNN/Stage2/'

# Prefix for PaddleOCR (no extension)
SVTR_CKPT = SVTR_CKPT_DIR / 'best_accuracy'
CRNN_CKPT = CRNN_CKPT_DIR / 'best_accuracy'

# Working directories
WORK_DIR = Path('/kaggle/working')
DICT_PATH = WORK_DIR / 'vietnamese_dict.txt'
APP_PATH = WORK_DIR / 'Recognizing-Vietnamese-handwriting' / 'app.py'
TEMP_DIR = WORK_DIR / 'temp_infer'
TEMP_DIR.mkdir(exist_ok=True)

# Config files
SVTR_CFG = WORK_DIR / 'rec_svtr_stage2.yml'
CRNN_CFG = WORK_DIR / 'rec_crnn_stage2.yml'

# Debug information
print('Paths:')
print(f'DATA_ROOT     : {DATA_ROOT}')
print(f'TRAIN_DATA    : {TRAIN_DATA}')
print(f'TEST_DATA     : {TEST_DATA}')
print(f'SVTR_CKPT_DIR : {SVTR_CKPT_DIR}')
print(f'SVTR_CKPT     : {SVTR_CKPT}')
print(f'CRNN_CKPT_DIR : {CRNN_CKPT_DIR}')
print(f'CRNN_CKPT     : {CRNN_CKPT}')
print(f'DICT_PATH     : {DICT_PATH}')

# Verify checkpoint existence
print('\nCheckpoint verification:')
print('SVTR .pdparams exists:', (SVTR_CKPT.with_suffix('.pdparams')).exists())
print('CRNN .pdparams exists:', (CRNN_CKPT.with_suffix('.pdparams')).exists())


Paths:
DATA_ROOT     : /kaggle/input/datasets/khangphantrnvn/uit-handwriten-dataset/UIT_HWDB_line_converted
TRAIN_DATA    : /kaggle/input/datasets/khangphantrnvn/uit-handwriten-dataset/UIT_HWDB_line_converted/train_data
TEST_DATA     : /kaggle/input/datasets/khangphantrnvn/uit-handwriten-dataset/UIT_HWDB_line_converted/test_data
SVTR_CKPT_DIR : /kaggle/input/models/thoandanh/svtr-vietnamese-handwriten/pytorch/default/1/SVTR/Stage2/best_accuracy
SVTR_CKPT     : /kaggle/input/models/thoandanh/svtr-vietnamese-handwriten/pytorch/default/1/SVTR/Stage2/best_accuracy/best_accuracy
CRNN_CKPT_DIR : /kaggle/input/models/thoandanh/crnn-vietnamese-handwriten/pytorch/default/1/CRNN/Stage2
CRNN_CKPT     : /kaggle/input/models/thoandanh/crnn-vietnamese-handwriten/pytorch/default/1/CRNN/Stage2/best_accuracy
DICT_PATH     : /kaggle/working/vietnamese_dict.txt

Checkpoint verification:
SVTR .pdparams exists: True
CRNN .pdparams exists: True


## 4. Tạo Dictionary Tiếng Việt từ Dataset

In [4]:
# Quét ký tự từ cả train + test labels để đảm bảo
# không bỏ sót ký tự nào trong bộ test khi inference.

def build_vietnamese_dict(*label_dirs: Path, out_path: Path) -> list:
    """
    Đọc file labels.txt từ các thư mục dataset UIT.
    Format gốc: images/xxx.jpg\tVăn bản tiếng Việt
    """
    char_set = set()
    total_lines = 0

    for d in label_dirs:
        label_file = d / 'labels.txt'
        if not label_file.exists():
            print(f'⚠️ Không tìm thấy: {label_file}')
            continue
        with open(label_file, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line or '\t' not in line:
                    continue
                _, text = line.split('\t', 1)
                char_set.update(text)
                total_lines += 1

    # Sắp xếp: ASCII trước → Unicode sau
    # (giống logic notebook train để index nhất quán)
    sorted_chars = sorted(char_set, key=lambda c: (ord(c) > 127, ord(c)))

    out_path.write_text('\n'.join(sorted_chars) + '\n', encoding='utf-8')

    print(f'✅ Dictionary: {len(sorted_chars):,} ký tự từ {total_lines:,} dòng label')
    print(f'   Lưu tại: {out_path}')
    print(f'   ASCII chars: {sum(1 for c in sorted_chars if ord(c) <= 127)}')
    print(f'   Unicode chars: {sum(1 for c in sorted_chars if ord(c) > 127)}')
    print(f'   Space: {" " in sorted_chars}')
    print(f'   Mẫu: {sorted_chars[:15]}')
    return sorted_chars

chars = build_vietnamese_dict(TRAIN_DATA, TEST_DATA, out_path=DICT_PATH)

✅ Dictionary: 161 ký tự từ 7,229 dòng label
   Lưu tại: /kaggle/working/vietnamese_dict.txt
   ASCII chars: 75
   Unicode chars: 86
   Space: True
   Mẫu: [' ', '!', '"', '%', '&', '(', ')', '*', ',', '-', '.', '/', '0', '1', '2']


## 5. Thiết lập Config inference cho SVTR và CRNN

In [5]:
import yaml

import yaml

svtr_cfg = {
    'Global': {
        'use_gpu': True,
        'character_dict_path': str(DICT_PATH),
        'character_type': 'ch',
        'max_text_length': 100,
        'use_space_char': True,
        'infer_mode': True,  
    },
    'Architecture': {
        'model_type': 'rec',
        'algorithm': 'SVTR',
        'Transform': None,
        'Backbone': {
            'name': 'SVTRNet',
            'img_size': [48, 800],
            'drop_rate': 0.1,
            'out_char_num': 200,
            'out_channels': 256,
            'patch_merging': 'Conv',
            'embed_dim': [128, 256, 384],
            'depth': [3, 6, 9],
            'num_heads': [4, 8, 12],
            'mixer': ['Local']*8 + ['Global']*10,
            'local_mixer': [[7, 11], [7, 11], [7, 11]],
            'last_stage': True,
            'prenorm': False,
        },
        'Neck': {'name': 'SequenceEncoder', 'encoder_type': 'reshape'},
        'Head': {'name': 'CTCHead'},
    },
    'Loss': {'name': 'CTCLoss'},
    'PostProcess': {'name': 'CTCLabelDecode'},
    'Metric': {'name': 'RecMetric', 'main_indicator': 'acc'},

    'Eval': {
        'dataset': {
            'name': 'SimpleDataSet',
            'data_dir': './',
            'transforms': [
                {'DecodeImage': {'img_mode': 'BGR', 'channel_first': False}},
                # ✅ FIX: Dùng RecResizeImg giống hệt training, KHÔNG dùng SVTRRecResizeImg
                {'RecResizeImg': {'image_shape': [3, 48, 800], 'padding': True}},
                {'KeepKeys': {'keep_keys': ['image']}}            ]
        }    }}
# ── Config CRNN (Dùng RecResizeImg thông thường) ──────────────
crnn_cfg = {
    'Global': {
        'use_gpu': True,
        'character_dict_path': str(DICT_PATH),
        'max_text_length': 100,
        'use_space_char': True,
        'infer_mode': False,
    },
    'Architecture': {
        'model_type': 'rec',
        'algorithm': 'CRNN',
        'Transform': None,
        'Backbone': {'name': 'ResNet', 'layers': 34},
        'Neck': {'name': 'SequenceEncoder', 'encoder_type': 'rnn', 'hidden_size': 256},
        'Head': {'name': 'CTCHead', 'fc_decay': 0},
    },
    'Loss': {'name': 'CTCLoss'},
    'PostProcess': {'name': 'CTCLabelDecode'},
    'Metric': {'name': 'RecMetric', 'main_indicator': 'acc'},
    
    'Eval': {
        'dataset': {
            'name': 'SimpleDataSet',
            'data_dir': './',
            'transforms': [
                {'DecodeImage': {'img_mode': 'BGR', 'channel_first': False}},
                {'RecResizeImg': {'image_shape': [3, 48, 800]}},
                {'KeepKeys': {'keep_keys': ['image']}}            ]
        }    }}
# Ghi file ra đĩa
SVTR_CFG.write_text(yaml.dump(svtr_cfg, allow_unicode=True), encoding='utf-8')
CRNN_CFG.write_text(yaml.dump(crnn_cfg, allow_unicode=True), encoding='utf-8')

print(f'✅ Đã cập nhật SVTR config tại: {SVTR_CFG}')
print(f'✅ Đã cập nhật CRNN config tại: {CRNN_CFG}')

✅ Đã cập nhật SVTR config tại: /kaggle/working/rec_svtr_stage2.yml
✅ Đã cập nhật CRNN config tại: /kaggle/working/rec_crnn_stage2.yml


## 6. Kiểm tra model checkpoint

In [6]:
# Cấu trúc: best_accuracy/ (thư mục) chứa best_accuracy.pdparams
import os

def check_checkpoint(ckpt_dir: Path, ckpt_prefix: Path, name: str):
    pdparams = Path(str(ckpt_prefix) + '.pdparams')
    if pdparams.exists():
        print(f'✅ {name}')
        print(f'   Thư mục : {ckpt_dir}')
        print(f'   Prefix  : {ckpt_prefix}  (dùng cho infer_rec.py)')
        for ext in ['.pdparams', '.pdopt', '.states']:
            f = Path(str(ckpt_prefix) + ext)
            if f.exists():
                print(f'   {f.name}  ({f.stat().st_size/1e6:.1f} MB)')
    else:
        print(f'❌ {name}: KHÔNG TÌM THẤY {pdparams}')
        print(f'   Hãy kiểm tra MODEL_SLUG và cấu trúc thư mục!')
        print(f'   Cấu trúc đúng: best_accuracy/best_accuracy.pdparams')

check_checkpoint(SVTR_CKPT_DIR, SVTR_CKPT, 'SVTR Stage2')
print()
check_checkpoint(CRNN_CKPT_DIR, CRNN_CKPT, 'CRNN Stage2')


✅ SVTR Stage2
   Thư mục : /kaggle/input/models/thoandanh/svtr-vietnamese-handwriten/pytorch/default/1/SVTR/Stage2/best_accuracy
   Prefix  : /kaggle/input/models/thoandanh/svtr-vietnamese-handwriten/pytorch/default/1/SVTR/Stage2/best_accuracy/best_accuracy  (dùng cho infer_rec.py)
   best_accuracy.pdparams  (92.1 MB)
   best_accuracy.pdopt  (0.3 MB)
   best_accuracy.states  (0.0 MB)

✅ CRNN Stage2
   Thư mục : /kaggle/input/models/thoandanh/crnn-vietnamese-handwriten/pytorch/default/1/CRNN/Stage2
   Prefix  : /kaggle/input/models/thoandanh/crnn-vietnamese-handwriten/pytorch/default/1/CRNN/Stage2/best_accuracy  (dùng cho infer_rec.py)
   best_accuracy.pdparams  (110.9 MB)
   best_accuracy.pdopt  (196.4 MB)
   best_accuracy.states  (0.0 MB)


## 7. Cài đặt package thư viện đồ hoạ

In [7]:
!pip install opencv-python-headless

## 8. Khởi động Streamlit Server qua Ngrok

In [11]:
import subprocess, time, os, sys
from pyngrok import ngrok

# ── 1) Điền ngrok authtoken ─────────────────────────

# Hoặc dùng Kaggle Secrets (an toàn hơn):
from kaggle_secrets import UserSecretsClient
NGROK_TOKEN = UserSecretsClient().get_secret("NGROK_TOKEN")

ngrok.set_auth_token(NGROK_TOKEN)

# ── 2) Chạy Streamlit trong background ──────────────────────
PORT = 8501
import os
env = os.environ.copy()
env['PADDLEOCR_DIR'] = str(PADDLEOCR_DIR)
env['WORK_DIR']      = str(WORK_DIR)
env['DICT_PATH']     = str(DICT_PATH)
env['SVTR_CKPT']     = str(SVTR_CKPT)
env['CRNN_CKPT']     = str(CRNN_CKPT)
env['SVTR_CFG']      = str(SVTR_CFG)
env['CRNN_CFG']      = str(CRNN_CFG)
env['TEMP_DIR']      = str(TEMP_DIR)


streamlit_proc = subprocess.Popen(
    [
        sys.executable, '-m', 'streamlit', 'run',
        str(APP_PATH),
        '--server.port', str(PORT),
        '--server.headless', 'true',
        '--server.enableCORS', 'false',
        '--server.enableXsrfProtection', 'false',
        '--browser.gatherUsageStats', 'false',
    ],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

print(f'⏳ Đang khởi động Streamlit (port {PORT})...')
time.sleep(6)   # đợi server ready

if streamlit_proc.poll() is not None:
    err = streamlit_proc.stderr.read().decode('utf-8')[-1000:]
    print(f'❌ Streamlit không chạy được:\n{err}')
else:
    # ── 3) Tạo ngrok tunnel ───────────────────────────────────
    public_url = ngrok.connect(PORT)
    print('\n' + '='*55)
    print(f'  🌐 DEMO URL: {public_url}')
    print('='*55)
    print('  Chia sẻ URL này để xem demo từ bất kỳ trình duyệt!')
    print('  URL hợp lệ trong suốt phiên Kaggle này.')
    print('='*55)

⏳ Đang khởi động Streamlit (port 8501)...                                                           

  🌐 DEMO URL: NgrokTunnel: "https://cindery-elmo-nonmonarchically.ngrok-free.dev" -> "http://localhost:8501"
  Chia sẻ URL này để xem demo từ bất kỳ trình duyệt!
  URL hợp lệ trong suốt phiên Kaggle này.


In [ ]:
# Chạy lệnh này trong một cell mới
!fuser -k 8501/tcp

## 9. (Tuỳ chọn) Test inference trực tiếp bằng CLI

In [ ]:
# Dùng để debug nếu inference trong app bị lỗi
import subprocess, sys, os, re, glob

def test_inference(img_path: str, model: str = 'svtr'):
    """
    Chạy infer_rec.py trực tiếp, in kết quả ra console.
    model: 'svtr' hoặc 'crnn'
    """
    if model == 'svtr':
        ckpt, cfg, shape = str(SVTR_CKPT), str(SVTR_CFG), '3,48,800'
    else:
        ckpt, cfg, shape = str(CRNN_CKPT), str(CRNN_CFG), '3,32,640'

    cmd = [
        sys.executable, 'tools/infer_rec.py',
        '-c', cfg,
        '-o',
        f'Global.pretrained_model={ckpt}',
        f'Global.infer_img={img_path}',
        f'Global.character_dict_path={DICT_PATH}',
        'Global.use_space_char=True',
        f'Global.rec_image_shape={shape}',
    ]

    print(f'▶ {" ".join(cmd[-10:])}')
    result = subprocess.run(
        cmd, capture_output=True, text=True,
        cwd=str(PADDLEOCR_DIR), timeout=120
    )
    print('STDOUT:', result.stdout[-2000:])
    if result.returncode != 0:
        print('STDERR:', result.stderr[-1000:])

# ── Chạy test với 1 ảnh từ test set ─────────────────────────
test_images = glob.glob(str(TEST_DATA / 'images/*.jpg'))[:1]
if test_images:
    print(f'🧪 Test với ảnh: {test_images[0]}')
    test_inference(test_images[0], model='svtr')
else:
    print('⚠️ Không tìm thấy ảnh test. Kiểm tra đường dẫn TEST_DATA.')

## 10. Tắt Streamlit Server

In [ ]:
# Chạy khi muốn kết thúc demo
try:
    ngrok.kill()       # Đóng tất cả ngrok tunnels
    print('✅ Đã đóng ngrok')
except Exception as e:
    print(f'ngrok: {e}')

try:
    streamlit_proc.terminate()
    print('✅ Đã dừng Streamlit')
except Exception as e:
    print(f'streamlit: {e}')